# Práctica Semana 2 — SIS206 Programación Orientada a Objetos II

## Abstracción y Encapsulamiento Avanzado
### Invariantes, Propiedades y Diseño por Responsabilidades

**Universidad Nacional del Altiplano — Escuela Profesional de Ingeniería de Sistemas**
**Semestre 2026-II · Puno — Perú**

Docente: Dr. Aldo Hernán Zanabria Gálvez

---

### Contenido
| Actividad | Tema | Lenguaje |
|---|---|---|
| 1 | `@property` avanzado — invariantes con setters validados | Python |
| 2 | Propiedades calculadas y composición | Python |
| 3 | Encapsulamiento en C++17 — `const correctness` | C++17 |
| 4 | Tabla de equivalencias Python vs C++ | Análisis |
| 5 | Proyecto Integrador — 3 clases encapsuladas + UML | Python + C++17 |
| VI | Preguntas de reflexión | Análisis |


---
## ACTIVIDAD 1 — `@property` avanzado: invariantes con setters validados

**Invariantes de la clase `Curso`:**

| ID | Invariante | Dónde se garantiza |
|---|---|---|
| I1 | `codigo` cumple `^[A-Z]{2,4}\d{3}$` | setter `codigo` |
| I2 | `1 <= creditos <= 6` | setter `creditos` |
| I3 | `1 <= cupos_maximo <= 50` | setter `cupos_maximo` |
| I4 | `0 <= inscritos <= cupos_maximo` | `inscribir_estudiante()` / `retirar_estudiante()` |

El constructor **asigna a través de los setters** (`self.codigo = codigo`, no `self.__codigo = codigo`):
así un objeto nunca llega a existir en estado inválido.


In [ ]:
"""
Práctica Semana 2 — SIS206 POO II — UNAP Puno — 2026-II
ACTIVIDAD 1: Clase Curso con encapsulamiento estricto.

Invariantes de clase garantizados en todo momento:
  I1: codigo  cumple el patrón ^[A-Z]{2,4}\\d{3}$
  I2: 1 <= creditos <= 6
  I3: 1 <= cupos_maximo <= 50
  I4: 0 <= inscritos <= cupos_maximo
"""
import re


class Curso:
    _PATRON_CODIGO = re.compile(r"^[A-Z]{2,4}\d{3}$")
    _MIN_CREDITOS, _MAX_CREDITOS = 1, 6
    _MIN_CUPOS, _MAX_CUPOS = 1, 50

    def __init__(self, codigo: str, nombre: str, creditos: int, cupos_maximo: int):
        self.codigo = codigo                # usa setter (valida)
        self.nombre = nombre                # usa setter (valida)
        self.creditos = creditos            # usa setter (valida)
        self.cupos_maximo = cupos_maximo    # usa setter (valida)
        self.__inscritos = 0                # privado — sin setter público

    # ------------------------------------------------------------------
    # codigo : lectura / escritura validada
    # ------------------------------------------------------------------
    @property
    def codigo(self) -> str:
        return self.__codigo

    @codigo.setter
    def codigo(self, valor: str) -> None:
        if not isinstance(valor, str):
            raise ValueError("El codigo debe ser una cadena de texto")
        valor = valor.strip().upper()
        if not self._PATRON_CODIGO.match(valor):
            raise ValueError(
                f"Codigo '{valor}' invalido: se esperaba 2-4 letras mayusculas "
                f"seguidas de 3 digitos (ej. SIS206)"
            )
        self.__codigo = valor

    # ------------------------------------------------------------------
    # nombre : lectura / escritura validada (no vacío)
    # ------------------------------------------------------------------
    @property
    def nombre(self) -> str:
        return self.__nombre

    @nombre.setter
    def nombre(self, valor: str) -> None:
        if not isinstance(valor, str) or not valor.strip():
            raise ValueError("El nombre del curso no puede estar vacio")
        self.__nombre = valor.strip()

    # ------------------------------------------------------------------
    # creditos : rango [1, 6]
    # ------------------------------------------------------------------
    @property
    def creditos(self) -> int:
        return self.__creditos

    @creditos.setter
    def creditos(self, valor: int) -> None:
        if not isinstance(valor, int) or isinstance(valor, bool):
            raise ValueError("Los creditos deben ser un numero entero")
        if not (self._MIN_CREDITOS <= valor <= self._MAX_CREDITOS):
            raise ValueError(
                f"Creditos {valor} fuera de rango "
                f"[{self._MIN_CREDITOS},{self._MAX_CREDITOS}]"
            )
        self.__creditos = valor

    # ------------------------------------------------------------------
    # cupos_maximo : rango [1, 50] y nunca menor a los ya inscritos
    # ------------------------------------------------------------------
    @property
    def cupos_maximo(self) -> int:
        return self.__cupos_maximo

    @cupos_maximo.setter
    def cupos_maximo(self, valor: int) -> None:
        if not isinstance(valor, int) or isinstance(valor, bool):
            raise ValueError("Los cupos deben ser un numero entero")
        if not (self._MIN_CUPOS <= valor <= self._MAX_CUPOS):
            raise ValueError(
                f"Cupos {valor} fuera de rango "
                f"[{self._MIN_CUPOS},{self._MAX_CUPOS}]"
            )
        # Protege el invariante I4 al reducir el aforo de un curso en marcha
        inscritos = getattr(self, "_Curso__inscritos", 0)
        if valor < inscritos:
            raise ValueError(
                f"No se puede fijar cupos_maximo={valor}: ya hay {inscritos} inscritos"
            )
        self.__cupos_maximo = valor

    # ------------------------------------------------------------------
    # Propiedades de SOLO LECTURA / CALCULADAS
    # ------------------------------------------------------------------
    @property
    def inscritos(self) -> int:
        """Solo lectura — solo cambia por inscribir_estudiante()/retirar_estudiante()."""
        return self.__inscritos

    @property
    def cupos_disponibles(self) -> int:
        """Propiedad CALCULADA — sin setter."""
        return self.__cupos_maximo - self.__inscritos

    @property
    def esta_lleno(self) -> bool:
        """Propiedad calculada booleana."""
        return self.__inscritos >= self.__cupos_maximo

    @property
    def porcentaje_ocupacion(self) -> float:
        """Propiedad calculada: % de aforo usado, redondeado a 2 decimales."""
        return round(self.__inscritos / self.__cupos_maximo * 100, 2)

    # ------------------------------------------------------------------
    # Lógica de negocio (verifica precondiciones)
    # ------------------------------------------------------------------
    def inscribir_estudiante(self) -> None:
        """Controla el invariante: inscritos <= cupos_maximo."""
        if self.esta_lleno:
            raise RuntimeError(f"Curso {self.__codigo} sin cupos disponibles")
        self.__inscritos += 1

    def retirar_estudiante(self) -> None:
        """Precondición: debe existir al menos un inscrito (invariante inscritos >= 0)."""
        if self.__inscritos == 0:
            raise RuntimeError(f"Curso {self.__codigo} no tiene inscritos que retirar")
        self.__inscritos -= 1

    def __str__(self) -> str:
        return (f"{self.__codigo} - {self.__nombre}"
                f" ({self.__creditos} cr)"
                f" [{self.__inscritos}/{self.__cupos_maximo} inscritos]")

    def __repr__(self) -> str:
        return (f"Curso({self.__codigo!r}, {self.__nombre!r}, "
                f"{self.__creditos}, {self.__cupos_maximo})")


# --- Pruebas -----------------------------------------------------------
if __name__ == "__main__":
    print("=" * 62)
    print("ACTIVIDAD 1 — Clase Curso con encapsulamiento estricto")
    print("=" * 62)

    c = Curso("SIS206", "POO II", 4, 30)
    print(c)
    print(f"Cupos disponibles: {c.cupos_disponibles}")

    for _ in range(3):
        c.inscribir_estudiante()
    print(c)
    print(f"Ocupacion: {c.porcentaje_ocupacion}% | Lleno: {c.esta_lleno}")

    print("\n-- Invariantes: deben lanzar ValueError --------------------")
    for caso in [("X9", "nombre", 4, 30),
                 ("SIS206", "nombre", 0, 30),
                 ("SIS206", "nombre", 4, 0),
                 ("SIS206", "nombre", 7, 30),
                 ("sis206", "", 4, 30)]:
        try:
            Curso(*caso)
            print(f"  FALLO: {caso} no lanzo excepcion")
        except ValueError as ex:
            print(f"  Error esperado {caso}: {ex}")

    print("\n-- Propiedades calculadas: NO admiten asignacion -----------")
    for prop, val in [("cupos_disponibles", 99), ("esta_lleno", False), ("inscritos", 50)]:
        try:
            setattr(c, prop, val)
            print(f"  FALLO: se pudo asignar {prop}")
        except AttributeError as ex:
            print(f"  AttributeError esperado en '{prop}': {ex}")

    print("\n-- Invariante inscritos <= cupos_maximo --------------------")
    lleno = Curso("SIS101", "Algoritmica", 3, 2)
    lleno.inscribir_estudiante()
    lleno.inscribir_estudiante()
    try:
        lleno.inscribir_estudiante()
    except RuntimeError as ex:
        print(f"  RuntimeError esperado: {ex}")
    try:
        lleno.cupos_maximo = 1
    except ValueError as ex:
        print(f"  ValueError esperado al reducir aforo: {ex}")

    print("\n-- Name mangling ------------------------------------------")
    try:
        print(c.__codigo)
    except AttributeError as ex:
        print(f"  AttributeError esperado: {ex}")
    print(f"  Acceso 'forzado' (NO recomendado): {c._Curso__codigo}")


---
## ACTIVIDAD 2 — Propiedades calculadas y composición

`Docente` **compone** una lista de objetos `Curso`. Las propiedades `total_creditos`,
`total_inscritos`, `edad` y `carga_disponible` se **derivan del estado actual** en cada
invocación: no existe ningún atributo que las almacene, por lo que es imposible que queden
desincronizadas.

El getter `cursos` devuelve una **tupla** (copia inmutable) para que el cliente no pueda
hacer `doc.cursos.append(...)` y saltarse el invariante de los 3 cursos máximos.


In [ ]:
"""
Práctica Semana 2 — SIS206 POO II — UNAP Puno — 2026-II
ACTIVIDAD 2: Clase Docente — composición con Curso y propiedades calculadas.

Invariantes de clase:
  I1: dni tiene exactamente 8 dígitos
  I2: correo institucional termina en @unap.edu.pe
  I3: 0 <= len(cursos) <= _MAX_CURSOS (3)
  I4: fecha_nac es pasada y la edad resultante es >= 18
"""
import re
from datetime import date



class Docente:
    _MAX_CURSOS = 3           # invariante: un docente no puede tener > 3 cursos
    _DOMINIO = "@unap.edu.pe"
    _PATRON_DNI = re.compile(r"^\d{8}$")

    def __init__(self, dni: str, nombres: str, apellidos: str,
                 correo: str, fecha_nac: date):
        self.__dni = self._validar_dni(dni)
        self.__nombres = self._validar_texto(nombres, "nombres")
        self.__apellidos = self._validar_texto(apellidos, "apellidos")
        self.__correo = self._validar_correo(correo)
        self.__fecha_nac = self._validar_fecha(fecha_nac)
        self.__cursos: list[Curso] = []

    # ------------------------------------------------------------------
    # Validadores (responsabilidad: verificar invariantes de entrada)
    # ------------------------------------------------------------------
    @staticmethod
    def _validar_dni(dni: str) -> str:
        if not isinstance(dni, str) or not Docente._PATRON_DNI.match(dni.strip()):
            raise ValueError(f"DNI '{dni}' invalido: debe tener exactamente 8 digitos")
        return dni.strip()

    @staticmethod
    def _validar_correo(correo: str) -> str:
        if not isinstance(correo, str):
            raise ValueError("El correo debe ser una cadena de texto")
        correo = correo.strip().lower()
        if not correo.endswith(Docente._DOMINIO) or len(correo) <= len(Docente._DOMINIO):
            raise ValueError(
                f"Correo '{correo}' invalido: debe terminar en {Docente._DOMINIO}"
            )
        return correo

    @staticmethod
    def _validar_texto(valor: str, campo: str) -> str:
        if not isinstance(valor, str) or not valor.strip():
            raise ValueError(f"El campo '{campo}' no puede estar vacio")
        return valor.strip()

    @staticmethod
    def _validar_fecha(fecha_nac: date) -> date:
        if not isinstance(fecha_nac, date):
            raise ValueError("fecha_nac debe ser un objeto datetime.date")
        if fecha_nac >= date.today():
            raise ValueError("La fecha de nacimiento debe ser anterior a hoy")
        return fecha_nac

    # ------------------------------------------------------------------
    # Getters de solo lectura (identidad: el DNI no se reescribe)
    # ------------------------------------------------------------------
    @property
    def dni(self) -> str:
        return self.__dni

    @property
    def nombres(self) -> str:
        return self.__nombres

    @property
    def apellidos(self) -> str:
        return self.__apellidos

    @property
    def fecha_nac(self) -> date:
        return self.__fecha_nac

    @property
    def cursos(self) -> tuple[Curso, ...]:
        """Devuelve una COPIA inmutable: impide que el cliente rompa el invariante I3."""
        return tuple(self.__cursos)

    # ------------------------------------------------------------------
    # correo : lectura / escritura validada
    # ------------------------------------------------------------------
    @property
    def correo(self) -> str:
        return self.__correo

    @correo.setter
    def correo(self, valor: str) -> None:
        self.__correo = self._validar_correo(valor)

    # ------------------------------------------------------------------
    # Propiedades CALCULADAS (derivadas del estado, nunca almacenadas)
    # ------------------------------------------------------------------
    @property
    def nombre_completo(self) -> str:
        return f"{self.__apellidos}, {self.__nombres}"

    @property
    def iniciales(self) -> str:
        return f"{self.__nombres[0]}.{self.__apellidos[0]}."

    @property
    def edad(self) -> int:
        """Calculada en tiempo real — no se almacena."""
        hoy = date.today()
        anios = hoy.year - self.__fecha_nac.year
        if (hoy.month, hoy.day) < (self.__fecha_nac.month, self.__fecha_nac.day):
            anios -= 1
        return anios

    @property
    def total_creditos(self) -> int:
        """Suma de creditos de todos los cursos asignados."""
        return sum(c.creditos for c in self.__cursos)

    @property
    def total_inscritos(self) -> int:
        """Total de estudiantes inscritos en todos sus cursos."""
        return sum(c.inscritos for c in self.__cursos)

    @property
    def carga_disponible(self) -> int:
        """Cursos que aun puede aceptar."""
        return self._MAX_CURSOS - len(self.__cursos)

    # ------------------------------------------------------------------
    # Lógica de negocio (verifica precondiciones)
    # ------------------------------------------------------------------
    def asignar_curso(self, curso: Curso) -> None:
        if not isinstance(curso, Curso):
            raise TypeError("Solo se pueden asignar objetos de tipo Curso")
        if len(self.__cursos) >= self._MAX_CURSOS:
            raise RuntimeError(
                f"{self.nombre_completo} ya tiene {self._MAX_CURSOS} cursos asignados"
            )
        if any(c.codigo == curso.codigo for c in self.__cursos):
            raise RuntimeError(f"El curso {curso.codigo} ya esta asignado")
        self.__cursos.append(curso)

    def __str__(self) -> str:
        return (f"Dr. {self.nombre_completo}"
                f" | Edad: {self.edad}"
                f" | Cursos: {len(self.__cursos)}"
                f" | Total creditos: {self.total_creditos}")


# --- Pruebas -----------------------------------------------------------
if __name__ == "__main__":
    print("=" * 62)
    print("ACTIVIDAD 2 — Docente: composicion y propiedades calculadas")
    print("=" * 62)

    doc = Docente("12345678", "Aldo Hernan", "Zanabria Galvez",
                  "azanabria@unap.edu.pe", date(1980, 5, 14))

    poo2 = Curso("SIS206", "POO II", 4, 30)
    poo2.inscribir_estudiante()
    poo2.inscribir_estudiante()
    doc.asignar_curso(poo2)

    print(doc)
    print(f"Total inscritos en sus cursos: {doc.total_inscritos}")
    print(f"Iniciales: {doc.iniciales} | Correo: {doc.correo}")

    print("\n-- Las propiedades calculadas siguen al estado real --------")
    bd = Curso("SIS301", "Base de Datos", 3, 25)
    for _ in range(5):
        bd.inscribir_estudiante()
    doc.asignar_curso(bd)
    print(f"  Tras asignar {bd.codigo}: creditos={doc.total_creditos}, "
          f"inscritos={doc.total_inscritos}, carga disponible={doc.carga_disponible}")
    poo2.inscribir_estudiante()
    print(f"  Tras 1 inscripcion mas en SIS206: inscritos={doc.total_inscritos} "
          f"(se recalcula, no se almacena)")

    print("\n-- Invariantes: validaciones del constructor ---------------")
    casos = [
        ("123", "A", "B", "a@unap.edu.pe", date(1980, 5, 14)),
        ("12345678", "A", "B", "a@gmail.com", date(1980, 5, 14)),
        ("12345678", "", "B", "a@unap.edu.pe", date(1980, 5, 14)),
        ("12345678", "A", "B", "a@unap.edu.pe", date(2090, 1, 1)),
    ]
    for caso in casos:
        try:
            Docente(*caso)
            print(f"  FALLO: {caso[0]}/{caso[3]} no lanzo excepcion")
        except ValueError as ex:
            print(f"  Error esperado: {ex}")

    print("\n-- Invariante: maximo 3 cursos -----------------------------")
    doc.asignar_curso(Curso("SIS404", "Ing. Software", 3, 20))
    try:
        doc.asignar_curso(Curso("SIS505", "Redes", 3, 20))
    except RuntimeError as ex:
        print(f"  RuntimeError esperado: {ex}")

    print("\n-- La lista interna esta protegida -------------------------")
    print(f"  type(doc.cursos) = {type(doc.cursos).__name__} (copia inmutable)")
    try:
        doc.cursos.append(Curso("SIS606", "IA", 3, 20))
    except AttributeError as ex:
        print(f"  AttributeError esperado: {ex}")
    print(f"  Cursos reales del docente: {len(doc.cursos)} (no se altero)")

    print("\n-- Propiedad calculada sin setter --------------------------")
    try:
        doc.nombre_completo = "otro"
    except AttributeError as ex:
        print(f"  AttributeError esperado: {ex}")


---
## ACTIVIDAD 3 — Encapsulamiento en C++17: `const correctness` y setters validados

Se replica `Curso` en C++17 con: atributos privados con sufijo `_`, setters que lanzan
`std::invalid_argument`, y **todos** los getters y propiedades calculadas marcados `const`.


In [ ]:
%%writefile Curso.hpp
// =====================================================================
// Practica Semana 2 - SIS206 POO II - UNAP Puno - 2026-II
// ACTIVIDAD 3: Curso.hpp - encapsulamiento C++17, const correctness
// Invariantes: codigo ~ [A-Z]{2,4}[0-9]{3} | 1<=creditos<=6
//              1<=cuposMaximo<=50 | 0<=inscritos<=cuposMaximo
// =====================================================================
#pragma once

#include <string>
#include <regex>
#include <stdexcept>
#include <iostream>
#include <utility>

class Curso {
    std::string codigo_;
    std::string nombre_;
    int creditos_ = 0;
    int cuposMaximo_ = 0;
    int inscritos_ = 0;              // inicializado en declaracion

    static const std::regex& patronCodigo() {
        static const std::regex patron(R"(^[A-Z]{2,4}[0-9]{3}$)");
        return patron;
    }

public:
    static constexpr int MIN_CREDITOS = 1;
    static constexpr int MAX_CREDITOS = 6;
    static constexpr int MIN_CUPOS    = 1;
    static constexpr int MAX_CUPOS    = 50;

    Curso(std::string codigo, std::string nombre, int creditos, int cuposMaximo) {
        setCodigo(codigo);           // validar en el constructor
        setNombre(std::move(nombre));
        setCreditos(creditos);
        setCuposMaximo(cuposMaximo);
    }

    // --- Setters validados -------------------------------------------
    void setCodigo(const std::string& cod) {
        if (!std::regex_match(cod, patronCodigo()))
            throw std::invalid_argument("Codigo invalido: '" + cod +
                                        "' (se espera 2-4 letras mayusculas + 3 digitos)");
        codigo_ = cod;
    }

    void setNombre(std::string nom) {
        if (nom.empty())
            throw std::invalid_argument("El nombre del curso no puede estar vacio");
        nombre_ = std::move(nom);
    }

    void setCreditos(int c) {
        if (c < MIN_CREDITOS || c > MAX_CREDITOS)
            throw std::invalid_argument("Creditos fuera de rango [1,6]");
        creditos_ = c;
    }

    void setCuposMaximo(int q) {
        if (q < MIN_CUPOS || q > MAX_CUPOS)
            throw std::invalid_argument("Cupos fuera de rango [1,50]");
        if (q < inscritos_)          // protege el invariante inscritos_ <= cuposMaximo_
            throw std::invalid_argument("Cupos menores a los estudiantes ya inscritos");
        cuposMaximo_ = q;
    }

    // --- Getters (const - no modifican el objeto) ---------------------
    const std::string& getCodigo() const { return codigo_; }
    const std::string& getNombre() const { return nombre_; }
    int getCreditos()    const { return creditos_; }
    int getCuposMaximo() const { return cuposMaximo_; }
    int getInscritos()   const { return inscritos_; }

    // --- Propiedades calculadas (const) ------------------------------
    int  cuposDisponibles() const { return cuposMaximo_ - inscritos_; }
    bool estaLleno()        const { return inscritos_ >= cuposMaximo_; }
    double porcentajeOcupacion() const {
        return 100.0 * inscritos_ / cuposMaximo_;
    }

    // --- Logica de negocio (NO const: modifica el estado) ------------
    void inscribirEstudiante() {
        if (estaLleno())
            throw std::runtime_error("Curso sin cupos: " + codigo_);
        ++inscritos_;
    }

    void retirarEstudiante() {
        if (inscritos_ == 0)
            throw std::runtime_error("Curso sin inscritos: " + codigo_);
        --inscritos_;
    }

    void imprimir() const {
        std::cout << codigo_ << " - " << nombre_
                  << " (" << creditos_ << " cr)"
                  << " [" << inscritos_ << "/" << cuposMaximo_ << "]\n";
    }
};


In [ ]:
%%writefile main.cpp
// =====================================================================
// Practica Semana 2 - Actividad 3: main.cpp - pruebas de Curso
// Compilar: g++ -std=c++17 -Wall -Wextra -o prog main.cpp
// =====================================================================
#include "Curso.hpp"
#include <iostream>

// Demuestra el beneficio de const correctness: esta funcion recibe una
// referencia constante y SOLO puede invocar metodos marcados const.
void reportar(const Curso& c) {
    std::cout << "  [reporte] " << c.getCodigo()
              << " ocupacion=" << c.porcentajeOcupacion() << "%"
              << " disponibles=" << c.cuposDisponibles()
              << " lleno=" << std::boolalpha << c.estaLleno() << "\n";
    // c.inscribirEstudiante();  // ERROR de compilacion: metodo no-const
}

int main() {
    std::cout << "==============================================\n";
    std::cout << "ACTIVIDAD 3 - Curso en C++17 (const correctness)\n";
    std::cout << "==============================================\n";

    Curso c("SIS206", "POO II", 4, 30);
    c.inscribirEstudiante();
    c.inscribirEstudiante();
    c.inscribirEstudiante();
    c.imprimir();
    reportar(c);

    std::cout << "\n-- Setters validados: std::invalid_argument --\n";
    try { Curso bad("SIS206", "bad", 0, 30); }
    catch (const std::invalid_argument& ex) { std::cout << "  " << ex.what() << "\n"; }

    try { Curso bad("X9", "bad", 4, 30); }
    catch (const std::invalid_argument& ex) { std::cout << "  " << ex.what() << "\n"; }

    try { Curso bad("SIS206", "bad", 4, 99); }
    catch (const std::invalid_argument& ex) { std::cout << "  " << ex.what() << "\n"; }

    try { c.setCuposMaximo(2); }
    catch (const std::invalid_argument& ex) { std::cout << "  " << ex.what() << "\n"; }

    std::cout << "\n-- Invariante inscritos <= cuposMaximo --\n";
    Curso lleno("SIS101", "Algoritmica", 3, 2);
    lleno.inscribirEstudiante();
    lleno.inscribirEstudiante();
    try { lleno.inscribirEstudiante(); }
    catch (const std::runtime_error& ex) { std::cout << "  " << ex.what() << "\n"; }

    std::cout << "\n-- Un objeto const solo admite metodos const --\n";
    const Curso fijo("SIS301", "Base de Datos", 3, 25);
    fijo.imprimir();
    reportar(fijo);
    // fijo.inscribirEstudiante();  // ERROR: descarta el calificador const
    // fijo.setCreditos(5);         // ERROR: descarta el calificador const

    std::cout << "\nOK: todas las pruebas ejecutadas.\n";
    return 0;
}


In [ ]:
!g++ -std=c++17 -Wall -Wextra -o prog main.cpp && ./prog

### Comprobación: `const` impide llamar métodos mutadores

La siguiente celda **debe fallar** al compilar. Ese es el resultado esperado.


In [ ]:
%%writefile prueba_const.cpp
#include "Curso.hpp"
void reportar(const Curso& c) { c.imprimir(); c.inscribirEstudiante(); }
int main() { const Curso f("SIS206","POO II",4,30); f.setCreditos(5); reportar(f); }


In [ ]:
!g++ -std=c++17 -Wall -Wextra -c prueba_const.cpp -o /dev/null

---
## ACTIVIDAD 4 — Tabla de equivalencias de encapsulamiento

| Concepto | Python | C++17 |
|---|---|---|
| Atributo privado | `__attr` (name mangling → `_Clase__attr`) | `tipo attr_;` bajo `private:` |
| Atributo protegido | `_attr` (convención, sin efecto en el intérprete) | `tipo attr_;` bajo `protected:` |
| Getter de solo lectura | `@property def attr(self)` | `const TipoR& getAttr() const` |
| Getter + setter validado | `@attr.setter def attr(self, v)` | `void setAttr(TipoR v)` + getter `const` |
| **Propiedad calculada (R/O)** | **`@property` sin `.setter` → `AttributeError` al asignar** | **método `const` sin atributo de respaldo: `int cuposDisponibles() const`** |
| **Método que NO modifica** | **convención + no toca `self.__x` (no hay verificación)** | **sufijo `const` en la firma: verificado por el compilador** |
| **Lanzar excepción de validación** | **`raise ValueError(...)` / `RuntimeError(...)`** | **`throw std::invalid_argument(...)` / `std::runtime_error(...)`** |

### Pregunta de análisis

> *En C++, un `const Curso` no puede llamar a `inscribirEstudiante()` aunque sea un método
> perfectamente válido en tiempo de ejecución. ¿Por qué el sistema de tipos impone esta
> restricción y cuál es el beneficio de diseño?*

**Respuesta.** El calificador `const` forma parte del **tipo** del puntero implícito `this`.
En un método no-const, `this` es `Curso*`; en uno `const`, es `const Curso*`. Al invocar
`fijo.inscribirEstudiante()` sobre un objeto `const`, el compilador tendría que convertir
`const Curso*` a `Curso*`, lo que **descarta un calificador** y está prohibido. El compilador
no razona sobre lo que el método hace en ejecución: razona sobre **lo que el método promete
en su firma**. `inscribirEstudiante()` no promete no modificar, así que se rechaza.

**Beneficios de diseño:**

1. **El contrato es verificable estáticamente.** `const` convierte "este método no modifica el
   objeto" de un comentario a una regla que el compilador hace cumplir. En Python la misma
   promesa solo existe por convención.
2. **Permite razonar localmente.** Si una función recibe `const Curso&`, quien la lee sabe —
   sin abrir su cuerpo — que el objeto sale igual que entró.
3. **Propaga la inmutabilidad.** Habilita objetos verdaderamente inmutables y su uso seguro
   en contextos compartidos o concurrentes (múltiples lectores sin sincronización).
4. **Detecta errores en compilación, no en producción.** Un `inscribirEstudiante()` llamado por
   error dentro de una rutina de reporte es un error de compilación, no un dato corrupto.
5. **Documenta la intención.** La firma distingue consultas (`const`) de comandos (no-`const`),
   que es exactamente la separación *query/command* del diseño por responsabilidades.


---
## ACTIVIDAD 5 — Proyecto Integrador: encapsulamiento de las 3 clases propias

**Sistema de Gestión Académica.** Las tres clases propias del proyecto, encapsuladas:

| Clase | Responsabilidad única | Atributos con invariante | Propiedad calculada | Método con precondición |
|---|---|---|---|---|
| `Calificacion` | Representar una nota ponderada válida | `__tipo`, `__valor`, `__peso`, `__recalificaciones` | `puntos_ponderados`, `literal`, `esta_aprobada` | `recalificar()` |
| `Estudiante` | Conocer identidad y avance curricular | `__codigo`, `__correo`, `__semestre`, `__creditos_aprobados` | `avance_curricular`, `condicion`, `es_egresable` | `aprobar_creditos()` |
| `Matricula` | Vincular Estudiante ↔ Curso y sus notas | `__estado`, `__calificaciones`, `__fecha` | `promedio`, `peso_acumulado`, `esta_aprobada` | `registrar_calificacion()`, `cerrar()`, `retirar()` |

### 5.1 Implementación en Python


In [ ]:
"""
Práctica Semana 2 — SIS206 POO II — UNAP Puno — 2026-II
ACTIVIDAD 5 — PROYECTO INTEGRADOR
Sistema de Gestión Académica — encapsulamiento estricto de las 3 clases propias:

  1) Calificacion  — nota ponderada de una evaluación
  2) Estudiante    — persona matriculable con avance curricular
  3) Matricula     — vínculo Estudiante ↔ Curso + sus calificaciones

Cada clase cumple el criterio de diseño exigido:
  · >= 3 atributos privados con invariante verificable
  · >= 1 propiedad calculada de solo lectura
  · >= 1 método de negocio que verifica una precondición
"""
import re
from datetime import date



# =====================================================================
# CLASE 1 — Calificacion
# Responsabilidad única: representar UNA nota válida y ponderada.
# Invariantes:
#   I1: 0.0 <= valor <= 20.0
#   I2: 0.0 <  peso  <= 1.0
#   I3: tipo pertenece a TIPOS_VALIDOS
#   I4: recalificaciones <= MAX_RECALIFICACIONES
# =====================================================================
class Calificacion:
    _MIN_VALOR, _MAX_VALOR = 0.0, 20.0
    _NOTA_APROBATORIA = 11.0
    _TIPOS_VALIDOS = ("PRACTICA", "PARCIAL", "FINAL", "TRABAJO")
    _MAX_RECALIFICACIONES = 1

    def __init__(self, tipo: str, valor: float, peso: float):
        self.__tipo = None
        self.__valor = None
        self.__peso = None
        self.__recalificaciones = 0
        self.tipo = tipo        # usa setter
        self.valor = valor      # usa setter
        self.peso = peso        # usa setter

    # --- tipo -------------------------------------------------------
    @property
    def tipo(self) -> str:
        return self.__tipo

    @tipo.setter
    def tipo(self, valor: str) -> None:
        if not isinstance(valor, str):
            raise ValueError("El tipo de evaluacion debe ser texto")
        v = valor.strip().upper()
        if v not in self._TIPOS_VALIDOS:
            raise ValueError(
                f"Tipo '{valor}' invalido: use uno de {self._TIPOS_VALIDOS}"
            )
        self.__tipo = v

    # --- valor ------------------------------------------------------
    @property
    def valor(self) -> float:
        return self.__valor

    @valor.setter
    def valor(self, v: float) -> None:
        if isinstance(v, bool) or not isinstance(v, (int, float)):
            raise ValueError("La calificacion debe ser numerica")
        if not (self._MIN_VALOR <= v <= self._MAX_VALOR):
            raise ValueError(
                f"Calificacion {v} fuera de rango "
                f"[{self._MIN_VALOR},{self._MAX_VALOR}]"
            )
        self.__valor = round(float(v), 2)

    # --- peso -------------------------------------------------------
    @property
    def peso(self) -> float:
        return self.__peso

    @peso.setter
    def peso(self, p: float) -> None:
        if isinstance(p, bool) or not isinstance(p, (int, float)):
            raise ValueError("El peso debe ser numerico")
        if not (0.0 < p <= 1.0):
            raise ValueError(f"Peso {p} fuera de rango (0.0, 1.0]")
        self.__peso = round(float(p), 4)

    # --- Propiedades CALCULADAS (solo lectura) -----------------------
    @property
    def puntos_ponderados(self) -> float:
        """Aporte real de esta nota al promedio final."""
        return round(self.__valor * self.__peso, 3)

    @property
    def esta_aprobada(self) -> bool:
        return self.__valor >= self._NOTA_APROBATORIA

    @property
    def literal(self) -> str:
        """Escala literal UNAP derivada del valor numerico."""
        v = self.__valor
        if v >= 18: return "AD (Logro destacado)"
        if v >= 14: return "A (Logro esperado)"
        if v >= 11: return "B (En proceso)"
        return "C (En inicio)"

    @property
    def recalificaciones(self) -> int:
        return self.__recalificaciones

    # --- Método de negocio con PRECONDICIÓN --------------------------
    def recalificar(self, nuevo_valor: float) -> None:
        """Precondición: no superar el máximo de recalificaciones permitidas."""
        if self.__recalificaciones >= self._MAX_RECALIFICACIONES:
            raise RuntimeError(
                f"La nota de {self.__tipo} ya fue recalificada "
                f"{self._MAX_RECALIFICACIONES} vez(ces): no admite otra revision"
            )
        self.valor = nuevo_valor        # reusa el setter -> invariante I1 intacto
        self.__recalificaciones += 1

    def __str__(self) -> str:
        return (f"{self.__tipo}: {self.__valor:.2f} "
                f"(peso {self.__peso:.0%}) -> {self.literal}")


# =====================================================================
# CLASE 2 — Estudiante
# Responsabilidad única: conocer la identidad y el avance curricular.
# Invariantes:
#   I1: codigo tiene exactamente 6 dígitos
#   I2: correo termina en @unap.edu.pe
#   I3: 1 <= semestre <= 10
#   I4: 0 <= creditos_aprobados <= TOTAL_CREDITOS_PLAN
# =====================================================================
class Estudiante:
    _PATRON_CODIGO = re.compile(r"^\d{6}$")
    _DOMINIO = "@unap.edu.pe"
    _MIN_SEMESTRE, _MAX_SEMESTRE = 1, 10
    TOTAL_CREDITOS_PLAN = 220

    def __init__(self, codigo: str, nombres: str, apellidos: str,
                 correo: str, semestre: int, creditos_aprobados: int = 0):
        self.__creditos_aprobados = 0
        self.codigo = codigo
        self.nombres = nombres
        self.apellidos = apellidos
        self.correo = correo
        self.semestre = semestre
        self.__creditos_aprobados = self._validar_creditos(creditos_aprobados)

    @classmethod
    def _validar_creditos(cls, c: int) -> int:
        if isinstance(c, bool) or not isinstance(c, int):
            raise ValueError("Los creditos aprobados deben ser un entero")
        if not (0 <= c <= cls.TOTAL_CREDITOS_PLAN):
            raise ValueError(
                f"Creditos aprobados {c} fuera de rango [0,{cls.TOTAL_CREDITOS_PLAN}]"
            )
        return c

    # --- codigo (identidad: setter validado) -------------------------
    @property
    def codigo(self) -> str:
        return self.__codigo

    @codigo.setter
    def codigo(self, valor: str) -> None:
        if not isinstance(valor, str) or not self._PATRON_CODIGO.match(valor.strip()):
            raise ValueError(f"Codigo '{valor}' invalido: debe tener 6 digitos")
        self.__codigo = valor.strip()

    # --- nombres / apellidos ----------------------------------------
    @property
    def nombres(self) -> str:
        return self.__nombres

    @nombres.setter
    def nombres(self, valor: str) -> None:
        if not isinstance(valor, str) or not valor.strip():
            raise ValueError("Los nombres no pueden estar vacios")
        self.__nombres = valor.strip().title()

    @property
    def apellidos(self) -> str:
        return self.__apellidos

    @apellidos.setter
    def apellidos(self, valor: str) -> None:
        if not isinstance(valor, str) or not valor.strip():
            raise ValueError("Los apellidos no pueden estar vacios")
        self.__apellidos = valor.strip().title()

    # --- correo ------------------------------------------------------
    @property
    def correo(self) -> str:
        return self.__correo

    @correo.setter
    def correo(self, valor: str) -> None:
        if not isinstance(valor, str):
            raise ValueError("El correo debe ser texto")
        v = valor.strip().lower()
        if not v.endswith(self._DOMINIO) or len(v) <= len(self._DOMINIO):
            raise ValueError(f"Correo '{valor}' invalido: debe terminar en {self._DOMINIO}")
        self.__correo = v

    # --- semestre ----------------------------------------------------
    @property
    def semestre(self) -> int:
        return self.__semestre

    @semestre.setter
    def semestre(self, valor: int) -> None:
        if isinstance(valor, bool) or not isinstance(valor, int):
            raise ValueError("El semestre debe ser un entero")
        if not (self._MIN_SEMESTRE <= valor <= self._MAX_SEMESTRE):
            raise ValueError(
                f"Semestre {valor} fuera de rango "
                f"[{self._MIN_SEMESTRE},{self._MAX_SEMESTRE}]"
            )
        self.__semestre = valor

    # --- Propiedades CALCULADAS (solo lectura) -----------------------
    @property
    def creditos_aprobados(self) -> int:
        """Solo lectura: solo cambia mediante aprobar_creditos()."""
        return self.__creditos_aprobados

    @property
    def nombre_completo(self) -> str:
        return f"{self.__apellidos}, {self.__nombres}"

    @property
    def avance_curricular(self) -> float:
        """% del plan de estudios completado — derivado, nunca almacenado."""
        return round(self.__creditos_aprobados / self.TOTAL_CREDITOS_PLAN * 100, 2)

    @property
    def es_egresable(self) -> bool:
        return self.__creditos_aprobados >= self.TOTAL_CREDITOS_PLAN

    @property
    def condicion(self) -> str:
        """Regular / Observado según avance esperado para su semestre."""
        esperado = self.TOTAL_CREDITOS_PLAN / self._MAX_SEMESTRE * (self.__semestre - 1)
        return "REGULAR" if self.__creditos_aprobados >= esperado * 0.7 else "OBSERVADO"

    # --- Método de negocio con PRECONDICIÓN --------------------------
    def aprobar_creditos(self, cantidad: int) -> None:
        """Precondición: no exceder el total de créditos del plan de estudios."""
        if isinstance(cantidad, bool) or not isinstance(cantidad, int) or cantidad <= 0:
            raise ValueError("La cantidad de creditos a aprobar debe ser un entero > 0")
        if self.__creditos_aprobados + cantidad > self.TOTAL_CREDITOS_PLAN:
            raise RuntimeError(
                f"{self.nombre_completo}: no puede superar los "
                f"{self.TOTAL_CREDITOS_PLAN} creditos del plan "
                f"(actual {self.__creditos_aprobados}, intento +{cantidad})"
            )
        self.__creditos_aprobados += cantidad

    def __str__(self) -> str:
        return (f"[{self.__codigo}] {self.nombre_completo} "
                f"| Sem. {self.__semestre} "
                f"| Avance: {self.avance_curricular}% "
                f"| {self.condicion}")


# =====================================================================
# CLASE 3 — Matricula
# Responsabilidad única: gestionar el vínculo Estudiante ↔ Curso
# y las calificaciones obtenidas en él.
# Invariantes:
#   I1: estado pertenece a ESTADOS_VALIDOS
#   I2: suma de pesos de las calificaciones <= 1.0
#   I3: len(calificaciones) <= MAX_EVALUACIONES
#   I4: fecha de matrícula no es futura
# =====================================================================
class Matricula:
    _ESTADOS_VALIDOS = ("ACTIVA", "RETIRADA", "CERRADA")
    _MAX_EVALUACIONES = 4
    _NOTA_APROBATORIA = 11.0

    def __init__(self, estudiante: Estudiante, curso: Curso,
                 fecha: date | None = None):
        if not isinstance(estudiante, Estudiante):
            raise TypeError("Se esperaba un objeto Estudiante")
        if not isinstance(curso, Curso):
            raise TypeError("Se esperaba un objeto Curso")
        fecha = fecha or date.today()
        if fecha > date.today():
            raise ValueError("La fecha de matricula no puede ser futura")

        self.__estudiante = estudiante
        self.__curso = curso
        self.__fecha = fecha
        self.__estado = "ACTIVA"
        self.__calificaciones: list[Calificacion] = []
        curso.inscribir_estudiante()     # el Curso protege su propio invariante

    # --- Getters de solo lectura -------------------------------------
    @property
    def estudiante(self) -> Estudiante:
        return self.__estudiante

    @property
    def curso(self) -> Curso:
        return self.__curso

    @property
    def fecha(self) -> date:
        return self.__fecha

    @property
    def estado(self) -> str:
        return self.__estado

    @property
    def calificaciones(self) -> tuple[Calificacion, ...]:
        """Copia inmutable: el cliente no puede romper los invariantes I2/I3."""
        return tuple(self.__calificaciones)

    # --- Propiedades CALCULADAS (solo lectura) -----------------------
    @property
    def peso_acumulado(self) -> float:
        return round(sum(c.peso for c in self.__calificaciones), 4)

    @property
    def promedio(self) -> float:
        """Promedio ponderado sobre el peso efectivamente evaluado."""
        if not self.__calificaciones:
            return 0.0
        return round(
            sum(c.puntos_ponderados for c in self.__calificaciones) / self.peso_acumulado,
            2
        )

    @property
    def esta_completa(self) -> bool:
        return abs(self.peso_acumulado - 1.0) < 1e-9

    @property
    def esta_aprobada(self) -> bool:
        return self.esta_completa and self.promedio >= self._NOTA_APROBATORIA

    @property
    def resumen(self) -> str:
        return (f"{self.__estudiante.codigo} @ {self.__curso.codigo} "
                f"-> {self.promedio:.2f} ({self.__estado})")

    # --- Métodos de negocio con PRECONDICIÓN -------------------------
    def registrar_calificacion(self, calificacion: Calificacion) -> None:
        """Precondiciones: matrícula ACTIVA, cupo de evaluaciones y peso total <= 1.0."""
        if not isinstance(calificacion, Calificacion):
            raise TypeError("Se esperaba un objeto Calificacion")
        if self.__estado != "ACTIVA":
            raise RuntimeError(
                f"No se registran notas en una matricula {self.__estado}"
            )
        if len(self.__calificaciones) >= self._MAX_EVALUACIONES:
            raise RuntimeError(
                f"Se alcanzo el maximo de {self._MAX_EVALUACIONES} evaluaciones"
            )
        if self.peso_acumulado + calificacion.peso > 1.0 + 1e-9:
            raise RuntimeError(
                f"El peso total superaria 1.0 "
                f"(acumulado {self.peso_acumulado}, nuevo {calificacion.peso})"
            )
        self.__calificaciones.append(calificacion)

    def cerrar(self) -> None:
        """Precondición: el 100% del peso debe estar evaluado."""
        if self.__estado != "ACTIVA":
            raise RuntimeError(f"La matricula ya esta {self.__estado}")
        if not self.esta_completa:
            raise RuntimeError(
                f"No se puede cerrar: peso evaluado {self.peso_acumulado:.2f} de 1.00"
            )
        self.__estado = "CERRADA"
        if self.promedio >= self._NOTA_APROBATORIA:
            self.__estudiante.aprobar_creditos(self.__curso.creditos)

    def retirar(self) -> None:
        """Precondición: solo una matrícula ACTIVA puede retirarse."""
        if self.__estado != "ACTIVA":
            raise RuntimeError(f"La matricula ya esta {self.__estado}")
        self.__estado = "RETIRADA"
        self.__curso.retirar_estudiante()

    def __str__(self) -> str:
        return (f"Matricula {self.__fecha} | {self.__estudiante.nombre_completo} "
                f"| {self.__curso.codigo} | Estado: {self.__estado} "
                f"| Promedio: {self.promedio:.2f} "
                f"| Aprobada: {self.esta_aprobada}")


# --- Pruebas -----------------------------------------------------------
if __name__ == "__main__":
    print("=" * 66)
    print("ACTIVIDAD 5 — PROYECTO INTEGRADOR: Calificacion, Estudiante, Matricula")
    print("=" * 66)

    print("\n### CLASE 1 — Calificacion ###")
    n1 = Calificacion("PRACTICA", 15.5, 0.30)
    print(f"  {n1}")
    print(f"  Puntos ponderados: {n1.puntos_ponderados} | Aprobada: {n1.esta_aprobada}")
    for caso in [("EXAMEN_SORPRESA", 15, 0.3), ("FINAL", 21, 0.3), ("FINAL", 15, 1.5)]:
        try:
            Calificacion(*caso)
        except ValueError as ex:
            print(f"  ValueError esperado: {ex}")
    n1.recalificar(17.0)
    print(f"  Tras recalificar: {n1}")
    try:
        n1.recalificar(19.0)
    except RuntimeError as ex:
        print(f"  RuntimeError esperado: {ex}")

    print("\n### CLASE 2 — Estudiante ###")
    est = Estudiante("205678", "juan carlos", "mamani quispe",
                     "jmamani@unap.edu.pe", 5, 96)
    print(f"  {est}")
    print(f"  Egresable: {est.es_egresable} | Avance: {est.avance_curricular}%")
    for caso in [("2056", "A", "B", "a@unap.edu.pe", 5),
                 ("205678", "A", "B", "a@hotmail.com", 5),
                 ("205678", "A", "B", "a@unap.edu.pe", 12)]:
        try:
            Estudiante(*caso)
        except ValueError as ex:
            print(f"  ValueError esperado: {ex}")
    try:
        est.aprobar_creditos(200)
    except RuntimeError as ex:
        print(f"  RuntimeError esperado: {ex}")
    try:
        est.avance_curricular = 100
    except AttributeError as ex:
        print(f"  AttributeError esperado (calculada): {ex}")

    print("\n### CLASE 3 — Matricula ###")
    poo2 = Curso("SIS206", "POO II", 4, 30)
    mat = Matricula(est, poo2)
    print(f"  {mat}")
    print(f"  Inscritos en el curso tras matricular: {poo2.inscritos}")

    mat.registrar_calificacion(Calificacion("PRACTICA", 16, 0.30))
    mat.registrar_calificacion(Calificacion("PARCIAL", 13, 0.30))
    print(f"  Peso acumulado: {mat.peso_acumulado} | Promedio parcial: {mat.promedio}")
    try:
        mat.cerrar()
    except RuntimeError as ex:
        print(f"  RuntimeError esperado: {ex}")
    try:
        mat.registrar_calificacion(Calificacion("FINAL", 18, 0.60))
    except RuntimeError as ex:
        print(f"  RuntimeError esperado: {ex}")

    mat.registrar_calificacion(Calificacion("FINAL", 14, 0.40))
    print(f"  Peso completo: {mat.esta_completa} | Promedio final: {mat.promedio}")
    creditos_antes = est.creditos_aprobados
    mat.cerrar()
    print(f"  {mat}")
    print(f"  Creditos del estudiante: {creditos_antes} -> {est.creditos_aprobados}")
    try:
        mat.registrar_calificacion(Calificacion("TRABAJO", 20, 0.1))
    except RuntimeError as ex:
        print(f"  RuntimeError esperado: {ex}")
    try:
        mat.retirar()
    except RuntimeError as ex:
        print(f"  RuntimeError esperado: {ex}")

    print("\n### Retiro de una matricula ACTIVA ###")
    est2 = Estudiante("205679", "Maria", "Condori Apaza",
                      "mcondori@unap.edu.pe", 3, 40)
    mat2 = Matricula(est2, poo2)
    print(f"  Inscritos: {poo2.inscritos}")
    mat2.retirar()
    print(f"  Tras retirar -> estado={mat2.estado}, inscritos={poo2.inscritos}")

    print("\nOK: proyecto integrador validado.")


### 5.2 Equivalente en C++17

In [ ]:
%%writefile Calificacion.hpp
// =====================================================================
// Proyecto Integrador - Semana 2 - Clase 1: Calificacion
// Invariantes: 0<=valor_<=20 | 0<peso_<=1 | tipo_ valido
//              recalificaciones_ <= MAX_RECALIFICACIONES
// =====================================================================
#pragma once

#include <string>
#include <array>
#include <algorithm>
#include <stdexcept>
#include <iostream>
#include <iomanip>

class Calificacion {
    std::string tipo_;
    double valor_ = 0.0;
    double peso_ = 0.0;
    int recalificaciones_ = 0;

public:
    static constexpr double MIN_VALOR = 0.0;
    static constexpr double MAX_VALOR = 20.0;
    static constexpr double NOTA_APROBATORIA = 11.0;
    static constexpr int MAX_RECALIFICACIONES = 1;

    Calificacion(const std::string& tipo, double valor, double peso) {
        setTipo(tipo);
        setValor(valor);
        setPeso(peso);
    }

    // --- Setters validados -------------------------------------------
    void setTipo(const std::string& t) {
        static const std::array<std::string, 4> validos =
            {"PRACTICA", "PARCIAL", "FINAL", "TRABAJO"};
        std::string up = t;
        std::transform(up.begin(), up.end(), up.begin(),
                       [](unsigned char ch) { return static_cast<char>(std::toupper(ch)); });
        if (std::find(validos.begin(), validos.end(), up) == validos.end())
            throw std::invalid_argument("Tipo de evaluacion invalido: " + t);
        tipo_ = up;
    }

    void setValor(double v) {
        if (v < MIN_VALOR || v > MAX_VALOR)
            throw std::invalid_argument("Calificacion fuera de rango [0,20]");
        valor_ = v;
    }

    void setPeso(double p) {
        if (p <= 0.0 || p > 1.0)
            throw std::invalid_argument("Peso fuera de rango (0,1]");
        peso_ = p;
    }

    // --- Getters const -----------------------------------------------
    const std::string& getTipo() const { return tipo_; }
    double getValor() const { return valor_; }
    double getPeso()  const { return peso_; }
    int getRecalificaciones() const { return recalificaciones_; }

    // --- Propiedades calculadas (const) ------------------------------
    double puntosPonderados() const { return valor_ * peso_; }
    bool estaAprobada() const { return valor_ >= NOTA_APROBATORIA; }
    std::string literal() const {
        if (valor_ >= 18) return "AD (Logro destacado)";
        if (valor_ >= 14) return "A (Logro esperado)";
        if (valor_ >= 11) return "B (En proceso)";
        return "C (En inicio)";
    }

    // --- Negocio con precondicion ------------------------------------
    void recalificar(double nuevoValor) {
        if (recalificaciones_ >= MAX_RECALIFICACIONES)
            throw std::runtime_error("La nota de " + tipo_ + " ya fue recalificada");
        setValor(nuevoValor);       // reusa el setter: invariante intacto
        ++recalificaciones_;
    }

    void imprimir() const {
        std::cout << tipo_ << ": " << std::fixed << std::setprecision(2) << valor_
                  << " (peso " << peso_ << ") -> " << literal() << "\n";
    }
};


In [ ]:
%%writefile Estudiante.hpp
// =====================================================================
// Proyecto Integrador - Semana 2 - Clase 2: Estudiante
// Invariantes: codigo_ = 6 digitos | correo_ termina en @unap.edu.pe
//              1<=semestre_<=10 | 0<=creditosAprobados_<=TOTAL_CREDITOS_PLAN
// =====================================================================
#pragma once

#include <string>
#include <regex>
#include <stdexcept>
#include <iostream>
#include <iomanip>

class Estudiante {
    std::string codigo_;
    std::string nombres_;
    std::string apellidos_;
    std::string correo_;
    int semestre_ = 1;
    int creditosAprobados_ = 0;

    static const std::regex& patronCodigo() {
        static const std::regex p(R"(^[0-9]{6}$)");
        return p;
    }

public:
    static constexpr int TOTAL_CREDITOS_PLAN = 220;
    static constexpr int MIN_SEMESTRE = 1;
    static constexpr int MAX_SEMESTRE = 10;

    Estudiante(const std::string& codigo, const std::string& nombres,
               const std::string& apellidos, const std::string& correo,
               int semestre, int creditosAprobados = 0) {
        setCodigo(codigo);
        setNombres(nombres);
        setApellidos(apellidos);
        setCorreo(correo);
        setSemestre(semestre);
        setCreditosAprobados(creditosAprobados);
    }

    // --- Setters validados -------------------------------------------
    void setCodigo(const std::string& c) {
        if (!std::regex_match(c, patronCodigo()))
            throw std::invalid_argument("Codigo invalido: debe tener 6 digitos");
        codigo_ = c;
    }

    void setNombres(const std::string& n) {
        if (n.empty()) throw std::invalid_argument("Los nombres no pueden estar vacios");
        nombres_ = n;
    }

    void setApellidos(const std::string& a) {
        if (a.empty()) throw std::invalid_argument("Los apellidos no pueden estar vacios");
        apellidos_ = a;
    }

    void setCorreo(const std::string& c) {
        const std::string dominio = "@unap.edu.pe";
        if (c.size() <= dominio.size() ||
            c.compare(c.size() - dominio.size(), dominio.size(), dominio) != 0)
            throw std::invalid_argument("Correo invalido: debe terminar en @unap.edu.pe");
        correo_ = c;
    }

    void setSemestre(int s) {
        if (s < MIN_SEMESTRE || s > MAX_SEMESTRE)
            throw std::invalid_argument("Semestre fuera de rango [1,10]");
        semestre_ = s;
    }

    void setCreditosAprobados(int c) {
        if (c < 0 || c > TOTAL_CREDITOS_PLAN)
            throw std::invalid_argument("Creditos aprobados fuera de rango [0,220]");
        creditosAprobados_ = c;
    }

    // --- Getters const -----------------------------------------------
    const std::string& getCodigo()    const { return codigo_; }
    const std::string& getNombres()   const { return nombres_; }
    const std::string& getApellidos() const { return apellidos_; }
    const std::string& getCorreo()    const { return correo_; }
    int getSemestre() const { return semestre_; }
    int getCreditosAprobados() const { return creditosAprobados_; }

    // --- Propiedades calculadas (const) ------------------------------
    std::string nombreCompleto() const { return apellidos_ + ", " + nombres_; }
    double avanceCurricular() const {
        return 100.0 * creditosAprobados_ / TOTAL_CREDITOS_PLAN;
    }
    bool esEgresable() const { return creditosAprobados_ >= TOTAL_CREDITOS_PLAN; }
    std::string condicion() const {
        double esperado = static_cast<double>(TOTAL_CREDITOS_PLAN) / MAX_SEMESTRE
                          * (semestre_ - 1);
        return (creditosAprobados_ >= esperado * 0.7) ? "REGULAR" : "OBSERVADO";
    }

    // --- Negocio con precondicion ------------------------------------
    void aprobarCreditos(int cantidad) {
        if (cantidad <= 0)
            throw std::invalid_argument("La cantidad de creditos debe ser > 0");
        if (creditosAprobados_ + cantidad > TOTAL_CREDITOS_PLAN)
            throw std::runtime_error(nombreCompleto() +
                ": no puede superar los 220 creditos del plan");
        creditosAprobados_ += cantidad;
    }

    void imprimir() const {
        std::cout << "[" << codigo_ << "] " << nombreCompleto()
                  << " | Sem. " << semestre_
                  << " | Avance: " << std::fixed << std::setprecision(2)
                  << avanceCurricular() << "%"
                  << " | " << condicion() << "\n";
    }
};


In [ ]:
%%writefile Matricula.hpp
// =====================================================================
// Proyecto Integrador - Semana 2 - Clase 3: Matricula
// Invariantes: estado_ valido | suma de pesos <= 1.0
//              calificaciones_.size() <= MAX_EVALUACIONES
// Nota: usa asociacion por referencia a Estudiante y Curso (no los posee).
// =====================================================================
#pragma once

#include "Curso.hpp"
#include "Estudiante.hpp"
#include "Calificacion.hpp"

#include <vector>
#include <string>
#include <stdexcept>
#include <numeric>
#include <cmath>
#include <iostream>
#include <iomanip>

class Matricula {
    Estudiante* estudiante_;            // asociacion: no gestiona su ciclo de vida
    Curso* curso_;
    std::string fecha_;
    std::string estado_ = "ACTIVA";
    std::vector<Calificacion> calificaciones_;

public:
    static constexpr std::size_t MAX_EVALUACIONES = 4;
    static constexpr double NOTA_APROBATORIA = 11.0;

    Matricula(Estudiante& estudiante, Curso& curso, std::string fecha)
        : estudiante_(&estudiante), curso_(&curso), fecha_(std::move(fecha)) {
        if (fecha_.empty())
            throw std::invalid_argument("La fecha de matricula no puede estar vacia");
        curso_->inscribirEstudiante();  // el Curso protege su propio invariante
    }

    // --- Getters const -----------------------------------------------
    const Estudiante& getEstudiante() const { return *estudiante_; }
    const Curso& getCurso() const { return *curso_; }
    const std::string& getFecha()  const { return fecha_; }
    const std::string& getEstado() const { return estado_; }
    const std::vector<Calificacion>& getCalificaciones() const { return calificaciones_; }

    // --- Propiedades calculadas (const) ------------------------------
    double pesoAcumulado() const {
        double s = 0.0;
        for (const auto& c : calificaciones_) s += c.getPeso();
        return s;
    }

    double promedio() const {
        if (calificaciones_.empty()) return 0.0;
        double puntos = 0.0;
        for (const auto& c : calificaciones_) puntos += c.puntosPonderados();
        return puntos / pesoAcumulado();
    }

    bool estaCompleta() const { return std::fabs(pesoAcumulado() - 1.0) < 1e-9; }
    bool estaAprobada() const { return estaCompleta() && promedio() >= NOTA_APROBATORIA; }

    // --- Negocio con precondiciones ----------------------------------
    void registrarCalificacion(const Calificacion& cal) {
        if (estado_ != "ACTIVA")
            throw std::runtime_error("No se registran notas en una matricula " + estado_);
        if (calificaciones_.size() >= MAX_EVALUACIONES)
            throw std::runtime_error("Se alcanzo el maximo de evaluaciones");
        if (pesoAcumulado() + cal.getPeso() > 1.0 + 1e-9)
            throw std::runtime_error("El peso total superaria 1.0");
        calificaciones_.push_back(cal);
    }

    void cerrar() {
        if (estado_ != "ACTIVA")
            throw std::runtime_error("La matricula ya esta " + estado_);
        if (!estaCompleta())
            throw std::runtime_error("No se puede cerrar: falta evaluar peso");
        estado_ = "CERRADA";
        if (promedio() >= NOTA_APROBATORIA)
            estudiante_->aprobarCreditos(curso_->getCreditos());
    }

    void retirar() {
        if (estado_ != "ACTIVA")
            throw std::runtime_error("La matricula ya esta " + estado_);
        estado_ = "RETIRADA";
        curso_->retirarEstudiante();
    }

    void imprimir() const {
        std::cout << "Matricula " << fecha_ << " | " << estudiante_->nombreCompleto()
                  << " | " << curso_->getCodigo()
                  << " | Estado: " << estado_
                  << " | Promedio: " << std::fixed << std::setprecision(2) << promedio()
                  << " | Aprobada: " << std::boolalpha << estaAprobada() << "\n";
    }
};


In [ ]:
%%writefile main_proyecto.cpp
// =====================================================================
// Proyecto Integrador - Semana 2 - Pruebas de las 3 clases en C++17
// Compilar: g++ -std=c++17 -Wall -Wextra -o proyecto main_proyecto.cpp
// =====================================================================
#include "Matricula.hpp"
#include <iostream>

int main() {
    std::cout << "======================================================\n";
    std::cout << "PROYECTO INTEGRADOR C++17 - Calificacion/Estudiante/Matricula\n";
    std::cout << "======================================================\n";

    std::cout << "\n### CLASE 1 - Calificacion ###\n";
    Calificacion n1("PRACTICA", 15.5, 0.30);
    n1.imprimir();
    std::cout << "  Puntos ponderados: " << n1.puntosPonderados() << "\n";
    try { Calificacion mala("SORPRESA", 15, 0.3); }
    catch (const std::invalid_argument& ex) { std::cout << "  " << ex.what() << "\n"; }
    try { Calificacion mala("FINAL", 21, 0.3); }
    catch (const std::invalid_argument& ex) { std::cout << "  " << ex.what() << "\n"; }
    n1.recalificar(17.0);
    std::cout << "  Tras recalificar: " << n1.getValor() << "\n";
    try { n1.recalificar(19.0); }
    catch (const std::runtime_error& ex) { std::cout << "  " << ex.what() << "\n"; }

    std::cout << "\n### CLASE 2 - Estudiante ###\n";
    Estudiante est("205678", "Juan Carlos", "Mamani Quispe",
                   "jmamani@unap.edu.pe", 5, 96);
    est.imprimir();
    try { Estudiante e("2056", "A", "B", "a@unap.edu.pe", 5); }
    catch (const std::invalid_argument& ex) { std::cout << "  " << ex.what() << "\n"; }
    try { Estudiante e("205678", "A", "B", "a@gmail.com", 5); }
    catch (const std::invalid_argument& ex) { std::cout << "  " << ex.what() << "\n"; }
    try { est.aprobarCreditos(200); }
    catch (const std::runtime_error& ex) { std::cout << "  " << ex.what() << "\n"; }

    std::cout << "\n### CLASE 3 - Matricula ###\n";
    Curso poo2("SIS206", "POO II", 4, 30);
    Matricula mat(est, poo2, "2026-09-17");
    mat.imprimir();
    std::cout << "  Inscritos en el curso: " << poo2.getInscritos() << "\n";

    mat.registrarCalificacion(Calificacion("PRACTICA", 16, 0.30));
    mat.registrarCalificacion(Calificacion("PARCIAL", 13, 0.30));
    std::cout << "  Peso acumulado: " << mat.pesoAcumulado()
              << " | Promedio parcial: " << mat.promedio() << "\n";
    try { mat.cerrar(); }
    catch (const std::runtime_error& ex) { std::cout << "  " << ex.what() << "\n"; }
    try { mat.registrarCalificacion(Calificacion("FINAL", 18, 0.60)); }
    catch (const std::runtime_error& ex) { std::cout << "  " << ex.what() << "\n"; }

    mat.registrarCalificacion(Calificacion("FINAL", 14, 0.40));
    int antes = est.getCreditosAprobados();
    mat.cerrar();
    mat.imprimir();
    std::cout << "  Creditos: " << antes << " -> " << est.getCreditosAprobados() << "\n";
    try { mat.retirar(); }
    catch (const std::runtime_error& ex) { std::cout << "  " << ex.what() << "\n"; }

    std::cout << "\n### Retiro de matricula ACTIVA ###\n";
    Estudiante est2("205679", "Maria", "Condori Apaza", "mcondori@unap.edu.pe", 3, 40);
    Matricula mat2(est2, poo2, "2026-09-17");
    std::cout << "  Inscritos: " << poo2.getInscritos() << "\n";
    mat2.retirar();
    std::cout << "  Tras retirar -> estado=" << mat2.getEstado()
              << ", inscritos=" << poo2.getInscritos() << "\n";

    std::cout << "\nOK: proyecto integrador C++ validado.\n";
    return 0;
}


In [ ]:
!g++ -std=c++17 -Wall -Wextra -o proyecto main_proyecto.cpp && ./proyecto

### 5.3 Diagrama de clases UML

Insertar aquí `uml_semana2.png` (exportado desde Draw.io a partir de `uml_semana2.drawio`):

```
from google.colab import files
files.upload()          # subir uml_semana2.png
```

![Diagrama UML Semana 2](uml_semana2.png)

El diagrama muestra visibilidades (`-` privado, `+` público, `/` derivado), tipos de todos
los atributos con su invariante entre llaves, firmas completas de métodos con parámetros y
tipo de retorno, y las relaciones: `Docente ◇— Curso` (agregación 1 a 0..3),
`Matricula —> Estudiante` y `Matricula —> Curso` (asociación), `Matricula ◆— Calificacion`
(composición 1 a 0..4).

### 5.4 Commit al repositorio

```bash
git add .
git commit -m "feat(semana2): encapsulamiento estricto e invariantes"
git push origin main
```


---
## VI. PREGUNTAS DE REFLEXIÓN

### 5. Atributo privado vs. atributo con invariante

Un atributo **simplemente privado** solo está oculto: impide el acceso desde fuera, pero
dentro de la clase cualquier método puede escribirlo sin control. El ocultamiento protege el
*acceso*, no el *contenido*. Un atributo **con invariante** canaliza toda escritura por un
único punto (el setter) que valida antes de asignar; el estado inválido es **inalcanzable**,
venga de donde venga la asignación. La diferencia es encapsulamiento superficial frente a
encapsulamiento profundo: el primero responde *"¿quién puede tocarlo?"* y el segundo
*"¿qué valores puede llegar a tener?"*.

**¿Cuándo saltarse la validación justificadamente?**

1. **Reconstrucción desde una fuente ya validada**: al cargar desde base de datos o
   deserializar miles de registros previamente verificados, revalidar duplica el costo; se
   usa un constructor alterno de confianza (`Curso._from_trusted(...)`).
2. **Estados transitorios internos dentro de una operación atómica**: durante un intercambio
   o reordenamiento, el objeto puede pasar por un estado intermedio inconsistente siempre que
   el invariante se restaure **antes** de que ningún método público retorne.
3. **Rutas críticas medidas**: cuando el perfilado demuestra que la validación domina un bucle
   caliente, y la entrada ya fue validada aguas arriba.

En los tres casos la justificación es la misma: alguien más **ya garantizó** el invariante.
Saltarse la validación "porque yo sé que el dato es correcto" no es justificación, es la causa
habitual de la corrupción de estado.

### 6. Ventaja de interfaz de `@property` frente a `setCreditos(4)`

La ventaja central es el **Principio de Acceso Uniforme** (Meyer): el cliente escribe
`c.creditos = 4` sin saber si detrás hay un campo simple, una validación o un cálculo. Esto
permite **evolucionar la implementación sin romper el código cliente**: una clase puede
publicarse con atributos públicos y, al descubrir después que necesita validación, convertirlos
en propiedades sin tocar ni una línea de quien la usa. En C++ esa misma decisión obliga a
cambiar `obj.creditos` por `obj.setCreditos(...)` en cada llamada del sistema.

Ventajas derivadas: sintaxis más legible y natural, homogeneidad entre atributos triviales y
validados (`c.nombre` y `c.creditos` se usan igual), y soporte directo de formas idiomáticas
como `c.creditos += 1`, que en C++ exige `c.setCreditos(c.getCreditos() + 1)`.

**Contrapeso honesto:** esa transparencia **oculta el costo y los efectos**. `c.creditos = 4`
parece una asignación barata y puede lanzar `ValueError` o ejecutar lógica pesada. El estilo
explícito de C++ hace visible en el sitio de llamada que se invoca un método con validación.
Es un intercambio entre ergonomía y transparencia, no una superioridad absoluta.

### 7. ¿Cuándo almacenar `cupos_disponibles` en lugar de calcularla?

**Calcularla** es correcto por defecto: siempre refleja el estado real y es imposible que se
desincronice. Almacenarla se justifica cuando:

- el cálculo es **caro** (agregaciones sobre colecciones grandes, consultas a BD, E/S) y se
  invoca con mucha más frecuencia de la que cambian sus entradas;
- el perfilado **demuestra** que es un cuello de botella real;
- se requiere **estabilidad temporal**: congelar el valor en un instante (una foto del aforo al
  cierre de matrícula) para auditoría o reporte;
- se necesita **ordenar o indexar** por ese valor en almacenamiento persistente.

Para `cupos_disponibles` —una resta entre dos enteros— nada de esto aplica: almacenarla sería
optimización prematura.

**Riesgo que introduce:** duplica la fuente de verdad y crea un **invariante adicional**
(`cupos_disponibles == cupos_maximo - inscritos`) que ahora hay que mantener manualmente en
*cada* punto que modifique `inscritos` o `cupos_maximo`. Basta que un método futuro olvide
actualizarla para que el objeto mienta sin lanzar ningún error: un fallo silencioso, difícil de
rastrear y que puede sobrevender un curso. Si la caché es imprescindible, debe invalidarse en
los setters (`self.__cache = None`) y recalcularse perezosamente, nunca actualizarse a mano.

### 8. ¿Por qué marcar los getters como `const` en C++?

Porque `const` no es decoración: es parte del **contrato verificable** del método y del tipo de
`this`. Un getter sin `const` es utilizable únicamente sobre objetos no-const, y esa restricción
se **propaga como un virus** por todo el sistema: obliga a que cualquier función que solo quiera
leer reciba una referencia no-const, y con ella el permiso implícito de modificar. Además, sin
`const` es imposible usar la clase en contenedores constantes, en miembros `const`, o compartirla
entre hilos como solo-lectura. Añadir `const` después, con la clase ya en uso, obliga a modificar
todas las firmas que la atraviesan.

**Ejemplo real de la Actividad 3.** La función de reporte recibe el curso por referencia constante:

```cpp
void reportar(const Curso& c) {
    std::cout << c.getCodigo() << " disponibles=" << c.cuposDisponibles() << "\n";
}
```

Si `getCodigo()` o `cuposDisponibles()` **no** estuvieran marcados `const`, esta función no
compilaría. El error real producido por `g++ -std=c++17 -Wall -Wextra` es:

```
error: passing 'const Curso' as 'this' argument discards qualifiers [-fpermissive]
note:   in call to 'void Curso::inscribirEstudiante()'
```

El mismo error aparece con `const Curso fijo(...); fijo.setCreditos(5);`. Es decir, omitir
`const` en los getters no es un detalle estilístico: **rompe la compilación** del primer punto
donde alguien intente leer el objeto sin permiso de escritura.


---
## VII. CONCLUSIONES

1. El encapsulamiento profundo no consiste en ocultar atributos, sino en **hacer inalcanzable
   el estado inválido**. La prueba práctica es que ningún `Curso` con 0 créditos llegó a
   construirse: la excepción se lanza dentro del constructor, antes de que el objeto exista.
2. Las propiedades calculadas (`cupos_disponibles`, `promedio`, `avance_curricular`) eliminan
   una clase entera de errores —la desincronización de datos derivados— a cambio de un costo de
   cómputo despreciable.
3. Python y C++17 alcanzan el mismo objetivo con filosofías distintas: Python confía en
   convenciones y ofrece sintaxis uniforme; C++ delega la verificación al compilador mediante el
   sistema de tipos y `const`. Ambas versiones del proyecto producen resultados idénticos.
4. Delimitar responsabilidades ahora (una razón de cambio por clase) es el paso previo al SRP de
   la Semana 9: `Curso` protege su aforo, `Matricula` la ponderación de notas, y ninguna invade
   los invariantes de la otra.

## VIII. REFERENCIAS

- Meyer, B. (1997). *Object-Oriented Software Construction* (2.ª ed.). Prentice Hall.
- Martin, R. C. (2008). *Clean Code: A Handbook of Agile Software Craftsmanship*. Prentice Hall.
- Stroustrup, B. (2024). *Programming: Principles and Practice Using C++* (3.ª ed.). Addison-Wesley.
- Python Software Foundation. (2026). *Descriptor HowTo Guide*. https://docs.python.org/3/howto/descriptor.html
- Cppreference. (2026). *const and volatile type qualifiers*. https://en.cppreference.com/w/cpp/language/cv
